# Lab 9.1 – Gegenrechnung zum Kostenbericht

**Beispiel (fertige Lösung).** Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

Die Gegenrechnung summiert `kostenstellen.csv` je Monat **ohne** das Skript des Skills und vergleicht das Ergebnis mit der Zeile `Summenkontrolle` im zuletzt erzeugten Kostenbericht. Gleiche Zahl auf zwei Wegen = der Bericht hat alle Buchungen erfasst.

In [1]:
import glob
import os
import re

import pandas as pd

DATEI = "data_kosten/kostenstellen.csv"                  # seit Arbeitsphase 1 in data_kosten/
if not os.path.exists(DATEI):
    DATEI = "lieferungen/kostenstellen.csv"              # falls die Datei noch nicht verschoben ist

berichte = sorted(glob.glob("berichte/kosten/bericht_*_v2.html"), key=os.path.getmtime)   # eigene Läufe
BERICHT = berichte[-1] if berichte else "project/examples/m09-final-project/l01-cost-report/berichte/kosten/bericht_2026_09_v2.html"
print("Datei:  ", DATEI)
print("Bericht:", BERICHT)

# Summenkontrolle aus dem Kasten "Prüfung" des Berichts lesen
html = open(BERICHT, encoding="utf-8").read()
zeile = re.search(r"Summenkontrolle: [^<]+", html).group(0)
monat, zeilen_bericht, summe_bericht = re.search(r"(\d{4}-\d{2}) – ([\d.]+) Zeilen, Summe \w+ ([\d.,]+)", zeile).groups()
zeilen_bericht = int(zeilen_bericht.replace(".", ""))
summe_bericht = float(summe_bericht.replace(".", "").replace(",", "."))
print(zeile)

Datei:   lieferungen/kostenstellen.csv
Bericht: project/examples/m09-final-project/l01-cost-report/berichte/kosten/bericht_2026_09_v2.html
Summenkontrolle: 2026-09 – 42 Zeilen, Summe Kosten 17.091,47 – stimmt mit der Kennzahlentabelle überein


## Schritt 1: Gegenrechnung je Monat (Aufgabe 1)

Die Zelle darunter mit dem vorgegebenen Prompt aus der Übung (Aufgabe 5) füllen, dann die Prüfzelle ausführen.

In [2]:
# >>> AUFGABE 1: kostenstellen.csv lesen, je Monat Anzahl Buchungen und Summe bilden – Prompt siehe Übung, Aufgabe 5
kosten = pd.read_csv(DATEI, sep=";", decimal=",", encoding="cp1252")
kosten["monat"] = pd.to_datetime(kosten["Buchungsdatum"], format="%d.%m.%Y").dt.strftime("%Y-%m")
je_monat = kosten.groupby("monat").agg(zeilen=("Betrag", "count"), summe=("Betrag", "sum"))
je_monat.tail(3)
# <<< ENDE AUFGABE 1

,zeilen,summe
monat,,
2026-07,52,19664.61
2026-08,46,24458.48
2026-09,42,17091.47


In [3]:
# Prüfzelle – nicht ändern
def deutsch(wert):
    return f"{wert:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pruefe_gegenrechnung():
    if "je_monat" not in globals():
        return "nicht OK – Tabelle je_monat fehlt noch (Aufgabe 1)."
    if not {"zeilen", "summe"} <= set(je_monat.columns):
        return "nicht OK – je_monat braucht die Spalten zeilen und summe."
    if monat not in je_monat.index:
        return f"nicht OK – Monat {monat} fehlt im Index von je_monat: Index = Monat im Format JJJJ-MM."
    if not pd.api.types.is_numeric_dtype(je_monat["summe"]):
        return "nicht OK – summe ist keine Zahl: beim Lesen decimal=\",\" setzen."
    anzahl, summe = int(je_monat.loc[monat, "zeilen"]), float(je_monat.loc[monat, "summe"])
    if anzahl != zeilen_bericht:
        return f"nicht OK – {anzahl} statt {zeilen_bericht} Buchungen im Monat {monat}."
    if abs(summe - summe_bericht) > 0.01:
        return f"nicht OK – Summe {deutsch(summe)} statt {deutsch(summe_bericht)} im Monat {monat}."
    return (f"OK – Gegenrechnung {monat}: {anzahl} Buchungen, Summe {deutsch(summe)} € "
            f"= Summenkontrolle des Berichts")


print(pruefe_gegenrechnung())

OK – Gegenrechnung 2026-09: 42 Buchungen, Summe 17.091,47 € = Summenkontrolle des Berichts


## Erweiterung: eigene CSV gegenrechnen

Für die eigene Datei oben `DATEI` auf den Pfad in `data_kosten/` setzen und in Aufgabe 1 Trennzeichen, Dezimalzeichen, Zeichensatz und Spaltennamen anpassen (siehe Übung, Erweiterung).